[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_target_selection.ipynb)

# Selecting essential targets in M. tuberculosis

**Orange group · Tuberculosis**

*Mycobacterium tuberculosis* has around 4,000 genes, and the group needs a shortlist
small enough to study properly. This notebook narrows them down to the genes that
matter in two strains of *M. tuberculosis* but not in a harmless relative.

## What you will do

- Load three genome-wide knockdown screens: two *M. tuberculosis* strains and one
  *M. smegmatis*, a relative that does not cause disease.
- Read the file's own description of its columns.
- Keep the genes that are essential and were measured reliably, one screen at a time.
- Keep what passes in both *M. tuberculosis* strains, then remove what is equally
  essential in *M. smegmatis*.
- Add a UniProt identifier to every target and download the shortlist.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the three screens

The data comes from [Bosch et al.,
2021](https://doi.org/10.1016/j.cell.2021.06.033), who used **CRISPRi** on every gene
in the genome. CRISPRi does not delete a gene, it turns its volume down. Using guides
of different strengths, the experiment can ask how much of a gene the bacterium can
afford to lose before it stops growing.

They ran it in three organisms: **H37Rv**, the reference *M. tuberculosis* strain;
**HN878**, a strain isolated from a patient; and ***M. smegmatis***, a fast-growing
relative that lives in soil and does not cause disease. All three are in one Excel
file, one sheet each.

First the packages and the file. The Setup cell above put the project folder in place, so the data is already here.

In [ ]:
import os

import pandas as pd
import stylia
from scripts import vulnerability

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()
RANDOM_SEED = 42
DATA = "data/mmc3.xlsx"

if not os.path.exists(DATA):
    raise FileNotFoundError(f"not found: {DATA}. Run the Setup cell again.")
print(f"reading {DATA}")

Read the three sheets. Each becomes a table of its own, and they stay separate until section 5.

In [ ]:
screens = {name: vulnerability.load_screen(DATA, name) for name in vulnerability.SCREENS}
pd.DataFrame([{"screen": name, "sheet": vulnerability.SCREENS[name],
               "genes": len(df), "columns": df.shape[1]}
              for name, df in screens.items()])

*M. smegmatis* has a bigger genome, which is why its screen covers more genes. Here
are the first rows of the H37Rv screen, with only the columns this notebook uses.

In [ ]:
KEY = ["locus_tag", "name", "crispr_ess", "certain", "vi", "vi_lower", "vi_upper"]
screens["H37Rv"][KEY].head()

## 2. What the columns say about each gene

Each row is one gene, described by 28 columns. You do not have to guess what they
mean: the file carries its own data dictionary in the `Legend` sheet, written by the
authors.

In [ ]:
legend = pd.read_excel(DATA, sheet_name="Legend", header=None,
                       names=["column", "description"]).dropna()
SHOWN = ["locus_tag", "name", "antibacterial", "tnseq_ess", "crispr_ess", "n_guides",
         "str_span", "certain", "Vulnerability Index", "VI Lower Bound", "VI Upper Bound"]
for _, row in legend.drop_duplicates("column").set_index("column").loc[SHOWN].iterrows():
    print(f"{row.name:>20}  {row['description']}")

Three things to add to those descriptions:

- **`locus_tag`** is the permanent identifier, like `Rv0667`, and it is what we join
  on later. The spreadsheet writes it as `RVBD0667`; `load_screen` has already put it
  in the standard form. **`name`** is the familiar name, like `rpoB`, and where a gene
  has none it is a copy of the locus tag.
- **`crispr_ess`** and **`tnseq_ess`** are two verdicts on the same question from two
  different experiments.
- **`antibacterial`** is filled in for only 18 genes. We never use it to choose
  anything, which makes it a fair check in section 7.

Comparing the two essentiality verdicts is worth doing before trusting either.

In [ ]:
pd.crosstab(screens["H37Rv"]["crispr_ess"], screens["H37Rv"]["tnseq_ess"])

> **Note:** TnSeq breaks genes completely rather than turning them down, and returns
> `Uncertain` or `Unknown` for nearly 200 genes. CRISPRi gives a verdict for every
> one, so this notebook follows `crispr_ess`.

Not every row is a protein-coding gene. The screen also covers ribosomal RNAs and
transfer RNAs, which are never made into protein and so cannot be drug targets here.
They are easy to spot: their identifier is not an `Rv` number.

In [ ]:
h37 = screens["H37Rv"]
not_genes = h37[~h37["locus_tag"].str.startswith("Rv")]
print(f"{len(not_genes)} of {len(h37):,} rows are not protein-coding genes")
not_genes[["locus_tag", "name", "crispr_ess", "vi"]].head()

## 3. Keep the essential genes, one screen at a time

A target has to be essential: if the bacterium grows perfectly well without the gene,
blocking its protein will not help.

Each screen is filtered on its own and they are **not** merged yet. The two
*M. tuberculosis* strains were grown in separate experiments and *M. smegmatis* is a
different species, so merging now would hide the disagreements we want to use later.

In [ ]:
counts = pd.DataFrame([{"screen": name, "genes": len(df),
                        "essential": int((df["crispr_ess"] == "Essential").sum())}
                       for name, df in screens.items()])
counts["share"] = (counts["essential"] / counts["genes"]).map("{:.1%}".format)
counts

About one gene in six is essential in *M. tuberculosis* and far fewer in
*M. smegmatis*. *M. tuberculosis* lives only inside a host and has lost many of the
genes a soil bacterium needs, so more of what it still has is indispensable.

## 4. Keep the genes that were measured reliably

Essential is a yes or no. The screen also gives each gene a **vulnerability index**:
the authors fitted a curve of how much the bacterium suffers as the gene is turned
down further and further, then added up the damage across the whole range.

**The more negative the number, the more vulnerable the gene.** A gene near -16 is
already in trouble when it is only partly switched off. A gene near -1 has to be shut
down almost completely before anything happens. That is the difference between a
target a real drug can work on and one it cannot, because no drug blocks all of its
target.

The **`certain`** column says whether that curve could be fitted reliably. Where it
could not, a vulnerability index is still printed, but its range is so wide the number
says nothing.

`add_selection_flags` turns each rule into a True/False column instead of removing rows, so we can count what every rule costs.

In [ ]:
flagged = {name: vulnerability.add_selection_flags(df) for name, df in screens.items()}
h37 = flagged["H37Rv"]
print(f"reliably measured: {h37['is_certain'].sum():,} of {len(h37):,} genes")
print(f"essential AND reliably measured: {h37['selected'].sum():,}")
pd.crosstab(h37["is_essential"], h37["is_certain"])

Only about one gene in seven was measured reliably, but essential genes are much more
likely to be among them: they are the ones where turning the gene down produced a
visible effect. The 552 in the bottom-right corner of that table are what we keep.

The plot below shows why the flag cannot be ignored.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for certain, color, label in [(False, nc.gray, "not reliable"), (True, nc.orange, "reliable")]:
    values = h37.loc[h37["is_certain"] == certain, "vi"]
    ax.hist(values, bins=60, range=(-20, 5), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{label} (n={len(values):,})")
ax.legend()
stylia.label(ax, xlabel="Vulnerability index", ylabel="Genes",
             title="Most genes in the screen were not measured reliably")

The unreliable genes are spread all over the scale, including the very vulnerable
end. Some may genuinely be vulnerable, but the experiment cannot tell us.

We do **not** put a cutoff on the vulnerability index itself. It is a smooth scale
with no natural dividing line, so any threshold would be our invention rather than a
result. It is used in section 8 to rank the shortlist, so the group can decide how far
down the list to look.

In [ ]:
steps = pd.DataFrame([{"screen": name,
                       "genes": len(df),
                       "essential": int(df["is_essential"].sum()),
                       "and reliably measured": int(df["selected"].sum())}
                      for name, df in flagged.items()])
steps

## 5. Genes that pass in both M. tuberculosis strains

H37Rv is a reference strain kept in laboratories for over a century. HN878 was
isolated from a patient and belongs to the W-Beijing family, which is widespread and
associated with drug resistance. A target that holds up in a clinical isolate as well
as the reference strain is a safer bet, so we keep the genes selected in **both**.

In [ ]:
selected = {name: set(df.loc[df["selected"], "locus_tag"])
            for name, df in flagged.items()}
both = selected["H37Rv"] & selected["HN878"]
print(f"H37Rv {len(selected['H37Rv'])}, HN878 {len(selected['HN878'])}, in both {len(both)}")
print(f"only H37Rv {len(selected['H37Rv'] - selected['HN878'])}, "
      f"only HN878 {len(selected['HN878'] - selected['H37Rv'])}")

Most genes agree across the two strains. Plotting one vulnerability index against the
other shows whether the genes we kept also rate similarly in both, which is a
different question from whether they were called essential.

In [ ]:
pair = flagged["H37Rv"].merge(flagged["HN878"], on="locus_tag", suffixes=("_h37rv", "_hn878"))
in_both = pair["locus_tag"].isin(both)
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for mask, color, label in [(~in_both, nc.gray, "not selected"), (in_both, nc.orange, "in both")]:
    ax.scatter(pair.loc[mask, "vi_h37rv"], pair.loc[mask, "vi_hn878"],
               color=color, alpha=0.5, s=8, label=f"{label} ({int(mask.sum()):,})")
ax.plot([-18, 3], [-18, 3], color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="H37Rv vulnerability index", ylabel="HN878 vulnerability index",
             title="The same gene in the reference and the clinical strain")

The selected genes sit close to the dashed line of equality, so a gene that is
vulnerable in one strain is generally vulnerable in the other.

## 6. Remove what is equally essential in M. smegmatis

*M. smegmatis* is a harmless soil-dwelling cousin. A gene that is essential there too
is likely one every bacterium needs, such as the machinery that builds proteins or
copies DNA. Those are real vulnerabilities but not *tuberculosis* vulnerabilities, and
a drug against one would be likely to hit harmless bacteria as well.

*M. smegmatis* genes have their own identifiers (`MSMEG0001` and so on), which say
nothing about the matching *M. tuberculosis* gene. What the two screens share is the
common gene name, so that is what we match on.

In [ ]:
core = flagged["H37Rv"][flagged["H37Rv"]["locus_tag"].isin(both)].copy()
msmeg_selected = set(flagged["Msmeg"].loc[flagged["Msmeg"]["selected"], "name"])
core["in_msmeg"] = core["name"].isin(msmeg_selected)
print(f"{core['in_msmeg'].sum()} of {len(core)} genes are essential in M. smegmatis too")
core.loc[core["in_msmeg"], ["locus_tag", "name", "vi"]].sort_values("vi").head(10)

The most vulnerable genes removed are ribosomal proteins (`rplF`, `rplE`, `rplB`,
`rpsC`), the protein export machinery (`secY`) and a transcription factor (`nusG`) —
exactly the kind of gene this comparison is meant to catch. That leaves the
shortlist.

In [ ]:
targets = core[~core["in_msmeg"]].copy()
print(f"{len(core)} genes in both strains -> {len(targets)} after removing "
      f"the M. smegmatis overlap")

Matching on names has a limit worth stating plainly, and it applies to the shortlist we just made.

In [ ]:
unnamed = targets["name"] == targets["locus_tag"]
print(f"{unnamed.sum()} of the {len(targets)} genes in the shortlist have no common "
      f"name, so they were never compared with M. smegmatis")
targets.loc[unnamed, ["locus_tag", "name", "crispr_ess", "vi"]].head()

> **Note:** those genes are in the shortlist because there was no evidence against
> them, not because they passed a test. Genes without a common name are usually the
> ones nobody has studied, which makes them interesting and risky in equal measure.

## 7. Check the shortlist against the drugs we already have

The screen labelled 18 genes with a drug that already targets them. Those drugs work,
so a sensible shortlist ought to contain most of them. We never used that column to
choose anything, so it is an independent check.

In [ ]:
known = h37[h37["antibacterial"].notna()].copy()
known["in_shortlist"] = known["locus_tag"].isin(targets["locus_tag"])
print(f"{known['in_shortlist'].sum()} of {len(known)} genes hit by a known TB drug "
      f"are in the shortlist")
known[["locus_tag", "name", "antibacterial", "vi", "is_essential", "is_certain",
       "in_shortlist"]].sort_values("vi")

Twelve of the eighteen are there, and the six that are not split into two groups:

- `atpE`, `ribD` and `23S` were not called essential, so they went in section 3.
- `gyrB`, `gyrA` and `rpoB` are essential and vulnerable, but were removed in section
  6 for being equally essential in *M. smegmatis*. DNA gyrase and RNA polymerase are
  universal bacterial targets, so this is the selectivity filter working, not failing.

What survives is dominated by the mycobacterial cell wall: `inhA` and `kasA` build
mycolic acids, `embA`, `embB` and `dprE1` build the arabinogalactan layer, `mmpL3`
exports mycolic acids. That is what the group is looking for.

> **Exercise:** the vulnerability index of these genes runs from `kasA` at -12.7 to
> `folC` at -2.6, and drugs exist against targets all along that range. If the group
> had kept only the most vulnerable genes, which of these drugs would it have missed?

## 8. Add a UniProt identifier to every target

The group's deliverable is a list of **proteins**, and the work that follows (finding
a structure, checking for known inhibitors, assessing druggability) is keyed on a
protein identifier. The standard is the **UniProt accession**, a code like `P9WGY9`.

Searching UniProt one gene name at a time is slow and unreliable: a name search
returns something for almost any query, often the wrong protein or the right protein
in the wrong strain. Instead we ask once for every protein in the *M. tuberculosis*
proteome and index it on UniProt's own `Rv` number, so the match is exact by
construction.

In [ ]:
lookup = vulnerability.uniprot_lookup()
print(f"{len(lookup):,} locus tags with a UniProt accession, from one request")
lookup.head(3)

Attach an accession to each target, and look at whatever fails to match rather than letting it disappear.

In [ ]:
annotated = targets.merge(lookup, on="locus_tag", how="left")
found = annotated["uniprot_ac"].notna()
print(f"{found.sum()} of {len(annotated)} targets have an accession")
annotated.loc[~found, ["locus_tag", "name", "vi"]]

The failures are the ribosomal and transfer RNAs from section 2. They have no UniProt
accession because they are never made into protein, so finding nothing is the right
answer and they can be dropped.

> **Note:** an empty result is not automatically a mistake, but it is always worth
> looking at. The reason can be anything from "this is not a protein" to "we joined on
> the wrong column".

In [ ]:
proteins = annotated[found].sort_values("vi")
COLUMNS = ["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi", "vi_lower",
           "vi_upper", "reviewed", "antibacterial"]
proteins = proteins[COLUMNS]
print(f"{len(proteins)} protein targets, most vulnerable first")
proteins.head()

Save it and download it, so the group has the file after Colab forgets this session.

In [ ]:
os.makedirs("outputs", exist_ok=True)
final_path = "outputs/mtb_selected_targets.csv"
proteins.to_csv(final_path, index=False)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(final_path)
print(f"{len(proteins)} targets written to {final_path}")

## Summary

- Starting from 4,052 genes in H37Rv: 737 are essential, 552 of those were also
  measured reliably, 512 of those pass in the HN878 clinical strain too, and 353
  remain once the 159 that are equally essential in *M. smegmatis* are removed. 348 of
  them have a UniProt accession; the 5 that do not are RNAs, not proteins.
- The vulnerability index says how far a gene must be turned down before the bacterium
  suffers. It ranks the shortlist rather than cutting it, because the genes with known
  TB drugs against them are spread right across the scale.
- 12 of the 18 genes with a known TB drug came through. The notable losses, `gyrA`,
  `gyrB` and `rpoB`, were removed for being essential in *M. smegmatis* too, which is
  the selectivity filter doing its job.
- 88 of the 353 have no common name, so they were never compared with *M. smegmatis*.
  Those are the genes nobody has studied.

**Next:** upload `mtb_selected_targets.csv` to the group's Drive folder so everyone
works from the same list. Which of these proteins has a known structure, and which
could a small molecule realistically bind?